In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy rustworkx scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Kwantowy algorytm przybliżonej optymalizacji

*Szacowane zużycie zasobów: 22 minuty na procesorze Heron r3 (UWAGA: To jest tylko szacunek. Twój czas wykonania może się różnić.)*
## Tło
Ten samouczek pokazuje, jak zaimplementować **Kwantowy Algorytm Przybliżonej Optymalizacji (QAOA)** – hybrydową (kwantowo-klasyczną) metodę iteracyjną – w kontekście wzorców Qiskit. Najpierw rozwiążesz problem **Maksymalnego Cięcia** (ang. *Max-Cut*) dla małego grafu, a następnie dowiesz się, jak wykonać go w skali użytkowej. Wszystkie wykonania sprzętowe w tym samouczku powinny zmieścić się w limicie czasu dla bezpłatnie dostępnego Planu Otwartego.

Problem Max-Cut jest problemem optymalizacyjnym, który jest trudny do rozwiązania (dokładniej, jest to problem *NP-trudny*) z wieloma zastosowaniami w klasteryzacji, nauce o sieciach i fizyce statystycznej. W tym samouczku rozważamy graf węzłów połączonych krawędziami i dążymy do podziału węzłów na dwa zbiory tak, aby liczba krawędzi przecinanych przez to cięcie była maksymalna.

![Ilustracja problemu max-cut](../docs/images/tutorials/quantum-approximate-optimization-algorithm/maxcut-illustration.avif)
## Wymagania
Zanim rozpoczniesz ten samouczek, upewnij się, że masz zainstalowane następujące elementy:
- Qiskit SDK w wersji 1.0 lub nowszej, z obsługą [wizualizacji](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime w wersji 0.22 lub nowszej (`pip install qiskit-ibm-runtime`)

Ponadto będziesz potrzebować dostępu do instancji na [IBM Quantum Platform](/guides/cloud-setup). Pamiętaj, że tego samouczka nie można wykonać w Planie Otwartym, ponieważ uruchamia on zadania z użyciem [sesji](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session), które są dostępne tylko w Planie Premium.
## Konfiguracja

In [1]:
import matplotlib
import matplotlib.pyplot as plt
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph
import numpy as np
from scipy.optimize import minimize
from collections import defaultdict
from typing import Sequence


from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler

## Część I. QAOA w małej skali
Pierwsza część tego samouczka wykorzystuje problem Max-Cut w małej skali, aby zilustrować kroki rozwiązywania problemu optymalizacyjnego z użyciem komputera kwantowego.

Zanim odwzorujemy ten problem na algorytm kwantowy, warto lepiej zrozumieć, jak problem Max-Cut staje się klasycznym kombinatorycznym problemem optymalizacyjnym. Zacznijmy od rozważenia minimalizacji funkcji $f(x)$

$$
\min_{x\in {0, 1}^n}f(x),
$$

gdzie wejście $x$ jest wektorem, którego składowe odpowiadają poszczególnym węzłom grafu. Następnie ograniczamy każdą z tych składowych do wartości $0$ lub $1$ (które reprezentują przynależność lub brak przynależności do cięcia). W tym przykładzie małej skali używamy grafu z $n=5$ węzłami.

Możemy zapisać funkcję dla pary węzłów $i,j$, która wskazuje, czy odpowiadająca krawędź $(i,j)$ należy do cięcia. Na przykład funkcja $x_i + x_j - 2 x_i x_j$ wynosi 1 tylko wtedy, gdy dokładnie jeden z elementów $x_i$ lub $x_j$ równa się 1 (co oznacza, że krawędź jest w cięciu), a w pozostałych przypadkach wynosi zero. Problem maksymalizacji krawędzi w cięciu można sformułować jako

$$
\max_{x\in {0, 1}^n} \sum_{(i,j)} x_i + x_j - 2 x_i x_j,
$$

co można przepisać jako minimalizację postaci

$$
\min_{x\in {0, 1}^n} \sum_{(i,j)}  2 x_i x_j - x_i - x_j.
$$

Minimum $f(x)$ w tym przypadku osiągane jest wtedy, gdy liczba krawędzi przecinanych przez cięcie jest maksymalna. Jak widać, nie ma tu jeszcze żadnego związku z obliczeniami kwantowymi. Musimy przeformułować ten problem na coś, co komputer kwantowy będzie w stanie zrozumieć.
Zainicjuj swój problem, tworząc graf z $n=5$ węzłami.

In [2]:
n = 5

graph = rx.PyGraph()
graph.add_nodes_from(np.arange(0, n, 1))
edge_list = [
    (0, 1, 1.0),
    (0, 2, 1.0),
    (0, 4, 1.0),
    (1, 2, 1.0),
    (2, 3, 1.0),
    (3, 4, 1.0),
]
graph.add_edges_from(edge_list)
draw_graph(graph, node_size=600, with_labels=True)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/6ced6bea-0.avif" alt="Output of the previous code cell" />

![Wynik poprzedniej komórki kodu](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/6ced6bea-0.avif)

### Krok 1: Odwzorowanie wejść klasycznych na problem kwantowy
Pierwszy krok wzorca polega na odwzorowaniu klasycznego problemu (grafu) na kwantowe **Circuit** i **operatory**. Aby to zrobić, należy wykonać trzy główne kroki:

1. Zastosować serię matematycznych przeformułowań, aby przedstawić ten problem w notacji problemów Kwadratowej Nieograniczonej Optymalizacji Binarnej (QUBO).
2. Przepisać problem optymalizacyjny jako Hamiltonian, dla którego stan podstawowy odpowiada rozwiązaniu minimalizującemu funkcję kosztu.
3. Stworzyć Circuit kwantowy, który będzie przygotowywał stan podstawowy tego Hamiltonianu za pomocą procesu podobnego do wyżarzania kwantowego.

**Uwaga:** W metodologii QAOA ostatecznie chcemy mieć operator (**Hamiltonian**) reprezentujący **funkcję kosztu** naszego algorytmu hybrydowego, a także sparametryzowany Circuit (**Ansatz**) reprezentujący stany kwantowe z kandydującymi rozwiązaniami problemu. Możemy próbkować z tych stanów kandydujących, a następnie oceniać je za pomocą funkcji kosztu.

#### Graf &rarr; problem optymalizacyjny
Pierwszym krokiem odwzorowania jest zmiana notacji. Poniżej wyrażamy problem w notacji QUBO:

$$
\min_{x\in {0, 1}^n}x^T Q x,
$$

gdzie $Q$ jest macierzą $n\times n$ liczb rzeczywistych, $n$ odpowiada liczbie węzłów w grafie, $x$ jest wektorem zmiennych binarnych wprowadzonym powyżej, a $x^T$ oznacza transpozycję wektora $x$.

```
Maximize
 -2*x_0*x_1 - 2*x_0*x_2 - 2*x_0*x_4 - 2*x_1*x_2 - 2*x_2*x_3 - 2*x_3*x_4 + 3*x_0
 + 2*x_1 + 3*x_2 + 2*x_3 + 2*x_4

Subject to
  No constraints

  Binary variables (5)
    x_0 x_1 x_2 x_3 x_4
```
### Problem optymalizacyjny &rarr; Hamiltonian
Następnie możemy przeformułować problem QUBO jako **Hamiltonian** (tutaj: macierz reprezentująca energię układu):

$$
H_C=\sum_{ij}Q_{ij}Z_iZ_j + \sum_i b_iZ_i.
$$

<details>
<summary>
**Kroki przeformułowania problemu QAOA do postaci Hamiltonianu**
</summary>

Aby pokazać, jak problem QAOA można przepisać w ten sposób, najpierw zastąp zmienne binarne $x_i$ nowym zestawem zmiennych $z_i\in{-1, 1}$ za pomocą

$$
x_i = \frac{1-z_i}{2}.
$$

Widać tutaj, że jeśli $x_i$ wynosi $0$, to $z_i$ musi wynosić $1$. Gdy $x_i$ są podstawione przez $z_i$ w problemie optymalizacyjnym ($x^TQx$), można uzyskać równoważne sformułowanie.

$$
x^TQx=\sum_{ij}Q_{ij}x_ix_j \\ =\frac{1}{4}\sum_{ij}Q_{ij}(1-z_i)(1-z_j) \\=\frac{1}{4}\sum_{ij}Q_{ij}z_iz_j-\frac{1}{4}\sum_{ij}(Q_{ij}+Q_{ji})z_i + \frac{n^2}{4}.
$$

Jeśli teraz zdefiniujemy $b_i=-\sum_{j}(Q_{ij}+Q_{ji})$, usuniemy współczynnik przedni i stały wyraz $n^2$, dochodzimy do dwóch równoważnych sformułowań tego samego problemu optymalizacyjnego.

$$
\min_{x\in{0,1}^n} x^TQx\Longleftrightarrow \min_{z\in{-1,1}^n}z^TQz + b^Tz
$$

Tutaj $b$ zależy od $Q$. Pamiętaj, że aby uzyskać $z^TQz + b^Tz$, pominęliśmy czynnik 1/4 i stałe przesunięcie $n^2$, które nie odgrywają roli w optymalizacji.

Teraz, aby uzyskać kwantowe sformułowanie problemu, awansuj zmienne $z_i$ do macierzy Pauliego $Z$, czyli macierzy $2\times 2$ postaci

$$
Z_i = \begin{pmatrix}1 & 0 \\ 0 & -1\end{pmatrix}.
$$

Gdy podstawiasz te macierze do powyższego problemu optymalizacyjnego, otrzymujesz następujący Hamiltonian

$$
H_C=\sum_{ij}Q_{ij}Z_iZ_j + \sum_i b_iZ_i.
$$

*Przypomnij też, że macierze $Z$ są osadzone w przestrzeni obliczeniowej komputera kwantowego, czyli przestrzeni Hilberta o rozmiarze $2^n\times 2^n$. Dlatego wyrażenia takie jak $Z_iZ_j$ należy rozumieć jako iloczyn tensorowy $Z_i\otimes Z_j$ osadzony w przestrzeni Hilberta $2^n\times 2^n$. Na przykład w problemie z pięcioma zmiennymi decyzyjnymi wyraz $Z_1Z_3$ oznacza $I\otimes Z_3\otimes I\otimes Z_1\otimes I$, gdzie $I$ jest macierzą jednostkową $2\times 2$.*
</details>

Ten Hamiltonian nazywany jest **Hamiltonianem funkcji kosztu**. Ma on właściwość, że jego stan podstawowy odpowiada rozwiązaniu **minimalizującemu funkcję kosztu $f(x)$**.
Dlatego, aby rozwiązać twój problem optymalizacyjny, musisz teraz przygotować stan podstawowy $H_C$ (lub stan o dużym pokryciu z nim) na komputerze kwantowym. Następnie próbkowanie z tego stanu będzie z dużym prawdopodobieństwem dawało rozwiązanie $min~f(x)$.
Rozważmy teraz Hamiltonian $H_C$ dla problemu **Max-Cut**. Niech każdy wierzchołek grafu będzie skojarzony z Qubit w stanie $|0\rangle$ lub $|1\rangle$, gdzie wartość oznacza zbiór, do którego należy wierzchołek. Celem problemu jest maksymalizacja liczby krawędzi $(v_1, v_2)$, dla których $v_1 = |0\rangle$ i $v_2 = |1\rangle$, lub odwrotnie. Jeśli skojarzymy operator $Z$ z każdym Qubit, gdzie

$$
    Z|0\rangle = |0\rangle \qquad Z|1\rangle = -|1\rangle
$$

to krawędź $(v_1, v_2)$ należy do cięcia, gdy wartość własna $(Z_1|v_1\rangle) \cdot (Z_2|v_2\rangle) = -1$; innymi słowy, Qubit skojarzone z $v_1$ i $v_2$ są różne. Analogicznie, $(v_1, v_2)$ nie należy do cięcia, gdy wartość własna $(Z_1|v_1\rangle) \cdot (Z_2|v_2\rangle) = 1$. Należy zauważyć, że nie interesuje nas dokładny stan Qubit skojarzony z każdym wierzchołkiem, lecz jedynie to, czy są one takie same, czy różne wzdłuż krawędzi. Problem Max-Cut wymaga znalezienia przypisania Qubit do wierzchołków takiego, aby wartość własna następującego Hamiltonianu była zminimalizowana
$$
    H_C = \sum_{(i,j) \in e} Q_{ij} \cdot Z_i Z_j.
$$

Innymi słowy, $b_i = 0$ dla wszystkich $i$ w problemie Max-Cut. Wartość $Q_{ij}$ oznacza wagę krawędzi. W tym samouczku rozważamy graf nieważony, tzn. $Q_{ij} = 1.0$ dla wszystkich $i, j$.

In [ ]:
def build_max_cut_paulis(
    graph: rx.PyGraph,
) -> list[tuple[str, list[int], float]]:
    """Convert the graph to Pauli list.

    This function does the inverse of `build_max_cut_graph`
    """
    pauli_list = []
    for edge in list(graph.edge_list()):
        weight = graph.get_edge_data(edge[0], edge[1])
        pauli_list.append(("ZZ", [edge[0], edge[1]], weight))
    return pauli_list


max_cut_paulis = build_max_cut_paulis(graph)
cost_hamiltonian = SparsePauliOp.from_sparse_list(max_cut_paulis, n)
print("Cost Function Hamiltonian:", cost_hamiltonian)

Cost Function Hamiltonian: SparsePauliOp(['IIIZZ', 'IIZIZ', 'ZIIIZ', 'IIZZI', 'IZZII', 'ZZIII'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])


#### Hamiltonian &rarr; quantum circuit

The Hamiltonian $H_C$ contains the quantum definition of your problem. Now you can create a quantum circuit that will help *sample* good solutions from the quantum computer. The QAOA is inspired by quantum annealing and applies alternating layers of operators in the quantum circuit.

The general idea is to start in the ground state of a known system, $H^{\otimes n}|0\rangle$ above, and then steer the system into the ground state of the cost operator that you are interested in. This is done by applying the operators $\exp\{-i\gamma_k H_C\}$ and $\exp\{-i\beta_k H_m\}$ with angles $\gamma_1,...,\gamma_p$ and $\beta_1,...,\beta_p~$.


The quantum circuit that you generate is **parametrized** by $\gamma_i$ and $\beta_i$, so you can try out different values of $\gamma_i$ and $\beta_i$ and sample from the resulting state.

![Circuit diagram with QAOA layers](../docs/images/tutorials/quantum-approximate-optimization-algorithm/circuit-diagram.svg)


In this case, you will try an example with one QAOA layer that contains two parameters: $\gamma_1$ and $\beta_1$.

In [4]:
circuit = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=2)
circuit.measure_all()

circuit.draw("mpl")

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/7bd8c6d4-f40f-4a11-a440-0b26d9021b53-0.avif" alt="Output of the previous code cell" />

In [5]:
circuit.parameters

ParameterView([ParameterVectorElement(β[0]), ParameterVectorElement(β[1]), ParameterVectorElement(γ[0]), ParameterVectorElement(γ[1])])

### Step 2: Optimize problem for quantum hardware execution

The circuit above contains a series of abstractions useful to think about quantum algorithms, but not possible to run on the hardware. To be able to run on a QPU, the circuit needs to undergo a series of operations that make up the **transpilation** or **circuit optimization** step of the pattern.

The Qiskit library offers a series of **transpilation passes** that cater to a wide range of circuit transformations. You need to make sure that your circuit is **optimized** for your purpose.

Transpilation may involves several steps, such as:

* **Initial mapping** of the qubits in the circuit (such as decision variables) to physical qubits on the device.
* **Unrolling** of the instructions in the quantum circuit to the hardware-native instructions that the backend understands.
* **Routing** of any qubits in the circuit that interact to physical qubits that are adjacent with one another.
* **Error suppression** by adding single-qubit gates to suppress noise with dynamical decoupling.


More information about transpilation is available in our [documentation](/docs/guides/transpile).

The following code transforms and optimizes the abstract circuit into a format that is ready for execution on one of devices accessible through the cloud using the **Qiskit IBM Runtime service**.

In [6]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
print(backend)

# Create pass manager for transpilation
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

candidate_circuit = pm.run(circuit)
candidate_circuit.draw("mpl", fold=False, idle_wires=False)

<IBMBackend('test_heron_pok_1')>


<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3f28a422-805c-4d3d-b5f6-62539e9133bd-1.avif" alt="Output of the previous code cell" />

### Step 3: Execute using Qiskit primitives

In the QAOA workflow, the optimal QAOA parameters are found in an iterative optimization loop, which runs a series of circuit evaluations and uses a classical optimizer to find the optimal $\beta_k$ and $\gamma_k$ parameters. This execution loop is executed via the following steps:

1. Define the initial parameters
2. Instantiate a new `Session` containing the optimization loop and the primitive used to sample the circuit
3. Once an optimal set of parameters is found, execute the circuit a final time to obtain a final distribution which will be used in the post-process step.

#### Define circuit with initial parameters
We start with arbitrary chosen parameters.

In [7]:
initial_gamma = np.pi
initial_beta = np.pi / 2
init_params = [initial_beta, initial_beta, initial_gamma, initial_gamma]

### Krok 2: Optymalizacja problemu pod kątem wykonania na sprzęcie kwantowym
Powyższy Circuit zawiera szereg abstrakcji przydatnych do myślenia o algorytmach kwantowych, ale niemożliwych do uruchomienia na sprzęcie. Aby móc uruchomić go na QPU, Circuit musi przejść przez serię operacji składających się na krok **transpilacji** lub **optymalizacji Circuit** wzorca.

Biblioteka Qiskit oferuje szereg **przebiegów transpilacji** obsługujących szerokie spektrum transformacji Circuit. Musisz upewnić się, że twój Circuit jest **zoptymalizowany** pod kątem twojego celu.

Transpilacja może obejmować kilka kroków, takich jak:

* **Wstępne mapowanie** Qubit w Circuit (np. zmiennych decyzyjnych) na fizyczne Qubit urządzenia.
* **Rozwijanie** instrukcji w Circuit kwantowym do instrukcji natywnych sprzętowi, które rozumie Backend.
* **Trasowanie** Qubit w Circuit, które ze sobą oddziałują, do fizycznych Qubit będących sąsiadami.
* **Tłumienie błędów** przez dodawanie bramek jednoQubit w celu tłumienia szumu za pomocą dynamicznego odsprzęgania.

Więcej informacji o transpilacji znajdziesz w naszej [dokumentacji](/guides/transpile).

Poniższy kod transformuje i optymalizuje abstrakcyjny Circuit do formatu gotowego do wykonania na jednym z urządzeń dostępnych przez chmurę przy użyciu usługi **Qiskit IBM Runtime**.

In [8]:
def cost_func_estimator(params, ansatz, hamiltonian, estimator):
    # transform the observable defined on virtual qubits to
    # an observable defined on all physical qubits
    isa_hamiltonian = hamiltonian.apply_layout(ansatz.layout)

    pub = (ansatz, isa_hamiltonian, params)
    job = estimator.run([pub])

    results = job.result()[0]
    cost = results.data.evs

    objective_func_vals.append(cost)

    return cost

In [9]:
objective_func_vals = []  # Global variable
with Session(backend=backend) as session:
    # If using qiskit-ibm-runtime<0.24.0, change `mode=` to `session=`
    estimator = Estimator(mode=session)
    estimator.options.default_shots = 1000

    # Set simple error suppression/mitigation options
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XY4"
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit, cost_hamiltonian, estimator),
        method="COBYLA",
        tol=1e-2,
    )
    print(result)

 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -1.6295230263157894
       x: [ 1.530e+00  1.439e+00  4.071e+00  4.434e+00]
    nfev: 26
   maxcv: 0.0


![Wynik poprzedniej komórki kodu](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3f28a422-805c-4d3d-b5f6-62539e9133bd-1.avif)

### Krok 3: Wykonanie z użyciem prymitywów Qiskit
W przepływie pracy QAOA optymalne parametry QAOA są znajdowane w iteracyjnej pętli optymalizacyjnej, która uruchamia serię ocen Circuit i używa klasycznego optymalizatora do znalezienia optymalnych parametrów $\beta_k$ i $\gamma_k$. Ta pętla wykonawcza jest realizowana w następujących krokach:

1. Zdefiniuj parametry początkowe
2. Utwórz nową `Session` zawierającą pętlę optymalizacyjną i prymityw użyty do próbkowania Circuit
3. Po znalezieniu optymalnego zestawu parametrów, wykonaj Circuit ostatni raz, aby uzyskać końcowy rozkład, który zostanie użyty w kroku post-przetwarzania.
#### Zdefiniuj Circuit z parametrami początkowymi
Zaczynamy od arbitralnie wybranych parametrów.

In [10]:
plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/e14ecc92-0.avif" alt="Output of the previous code cell" />

#### Zdefiniuj Backend i prymityw wykonawczy
Użyj **prymitywów Qiskit Runtime** do interakcji z backendami IBM&reg;. Dwa prymitywy to Sampler i Estimator, a wybór prymitywu zależy od rodzaju pomiaru, który chcesz wykonać na komputerze kwantowym. Do minimalizacji $H_C$ użyj Estimator, ponieważ pomiar funkcji kosztu to po prostu wartość oczekiwana $\langle H_C \rangle$.
#### Uruchom
Prymitywy oferują wiele [trybów wykonania](/guides/execution-modes) do planowania zadań na urządzeniach kwantowych, a przepływ pracy QAOA działa iteracyjnie w sesji.

![Ilustracja przedstawiająca zachowanie trybów uruchomieniowych: pojedyncze zadanie, Batch i Session.](../docs/images/tutorials/quantum-approximate-optimization-algorithm/runtime-modes.avif)

Możesz podłączyć funkcję kosztu opartą na Sampler do procedury minimalizacji SciPy, aby znaleźć optymalne parametry.

In [11]:
optimized_circuit = candidate_circuit.assign_parameters(result.x)
optimized_circuit.draw("mpl", fold=False, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/2989e76e-4296-4dd8-b065-2b8fced064cf-0.avif" alt="Output of the previous code cell" />

In [12]:
# If using qiskit-ibm-runtime<0.24.0, change `mode=` to `backend=`
sampler = Sampler(mode=backend)
sampler.options.default_shots = 10000

# Set simple error suppression/mitigation options
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XY4"
sampler.options.twirling.enable_gates = True
sampler.options.twirling.num_randomizations = "auto"

pub = (optimized_circuit,)
job = sampler.run([pub], shots=int(1e4))
counts_int = job.result()[0].data.meas.get_int_counts()
counts_bin = job.result()[0].data.meas.get_counts()
shots = sum(counts_int.values())
final_distribution_int = {key: val / shots for key, val in counts_int.items()}
final_distribution_bin = {key: val / shots for key, val in counts_bin.items()}
print(final_distribution_int)

{28: 0.0328, 11: 0.0343, 2: 0.0296, 25: 0.0308, 16: 0.0303, 27: 0.0302, 13: 0.0323, 7: 0.0312, 4: 0.0296, 9: 0.0295, 26: 0.0321, 30: 0.031, 23: 0.0324, 31: 0.0303, 21: 0.0335, 15: 0.0317, 12: 0.0309, 29: 0.0297, 3: 0.0313, 5: 0.0312, 6: 0.0274, 10: 0.0329, 22: 0.0353, 0: 0.0315, 20: 0.0326, 8: 0.0322, 14: 0.0306, 17: 0.0295, 18: 0.0279, 1: 0.0325, 24: 0.0334, 19: 0.0295}


### Step 4: Post-process and return result in desired classical format

The post-processing step interprets the sampling output to return a solution for your original problem. In this case, you are interested in the bitstring with the highest probability as this determines the optimal cut. The symmetries in the problem allow for four possible solutions, and the sampling process will return one of them with a slightly higher probability, but you can see in the plotted distribution below that four of the bitstrings are distinctively more likely than the rest.

In [13]:
# auxiliary functions to sample most likely bitstring
def to_bitstring(integer, num_bits):
    result = np.binary_repr(integer, width=num_bits)
    return [int(digit) for digit in result]


keys = list(final_distribution_int.keys())
values = list(final_distribution_int.values())
most_likely = keys[np.argmax(np.abs(values))]
most_likely_bitstring = to_bitstring(most_likely, len(graph))
most_likely_bitstring.reverse()

print("Result bitstring:", most_likely_bitstring)

Result bitstring: [0, 1, 1, 0, 1]


In [14]:
matplotlib.rcParams.update({"font.size": 10})
final_bits = final_distribution_bin
values = np.abs(list(final_bits.values()))
top_4_values = sorted(values, reverse=True)[:4]
positions = []
for value in top_4_values:
    positions.append(np.where(values == value)[0])
fig = plt.figure(figsize=(11, 6))
ax = fig.add_subplot(1, 1, 1)
plt.xticks(rotation=45)
plt.title("Result Distribution")
plt.xlabel("Bitstrings (reversed)")
plt.ylabel("Probability")
ax.bar(list(final_bits.keys()), list(final_bits.values()), color="tab:grey")
for p in positions:
    ax.get_children()[int(p[0])].set_color("tab:purple")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/650875e9-adbc-43bd-9505-556be2566278-0.avif" alt="Output of the previous code cell" />

![Wynik poprzedniej komórki kodu](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/e14ecc92-0.avif)

Po znalezieniu optymalnych parametrów dla Circuit możesz przypisać te parametry i próbkować końcowy rozkład uzyskany z zoptymalizowanymi parametrami. Tu właśnie należy użyć prymitywu *Sampler*, ponieważ interesuje nas rozkład prawdopodobieństwa pomiarów bitstring, który odpowiada optymalnemu cięciu grafu.

**Uwaga:** Oznacza to przygotowanie stanu kwantowego $\psi$ w komputerze, a następnie jego zmierzenie. Pomiar spowoduje kolaps stanu do pojedynczego stanu bazy obliczeniowej – na przykład `010101110000...` – który odpowiada kandydatowi na rozwiązanie $x$ naszego początkowego problemu optymalizacyjnego ($\max f(x)$ lub $\min f(x)$ w zależności od zadania).

In [15]:
# auxiliary function to plot graphs
def plot_result(G, x):
    colors = ["tab:grey" if i == 0 else "tab:purple" for i in x]
    pos, _default_axes = rx.spring_layout(G), plt.axes(frameon=True)
    rx.visualization.mpl_draw(
        G, node_color=colors, node_size=100, alpha=0.8, pos=pos
    )


plot_result(graph, most_likely_bitstring)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/33135970-8bc4-4fb2-ab87-08726a432ce4-0.avif" alt="Output of the previous code cell" />

And calculate the value of the cut:

In [16]:
def evaluate_sample(x: Sequence[int], graph: rx.PyGraph) -> float:
    assert len(x) == len(
        list(graph.nodes())
    ), "The length of x must coincide with the number of nodes in the graph."
    return sum(
        x[u] * (1 - x[v]) + x[v] * (1 - x[u])
        for u, v in list(graph.edge_list())
    )


cut_value = evaluate_sample(most_likely_bitstring, graph)
print("The value of the cut is:", cut_value)

The value of the cut is: 5


## Part II. Scale it up!

You have access to many devices with over 100 qubits on IBM Quantum&reg; Platform. Select one on which to solve Max-Cut on a 100-node weighted graph. This is a "utility-scale" problem. The steps to build the workflow are followed the same as above, but with a much larger graph.

In [17]:
n = 100  # Number of nodes in graph
graph_100 = rx.PyGraph()
graph_100.add_nodes_from(np.arange(0, n, 1))
elist = []
for edge in backend.coupling_map:
    if edge[0] < n and edge[1] < n:
        elist.append((edge[0], edge[1], 1.0))
graph_100.add_edges_from(elist)
draw_graph(graph_100, node_size=200, with_labels=True, width=1)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/590fe2ce-0.avif" alt="Output of the previous code cell" />

### Krok 4: Post-przetwarzanie i zwrócenie wyniku w żądanym formacie klasycznym
Krok post-przetwarzania interpretuje wynik próbkowania, aby zwrócić rozwiązanie twojego pierwotnego problemu. W tym przypadku interesuje cię bitstring o najwyższym prawdopodobieństwie, ponieważ to on wyznacza optymalne cięcie. Symetrie w problemie dopuszczają cztery możliwe rozwiązania, a proces próbkowania zwróci jedno z nich z nieco wyższym prawdopodobieństwem, ale na poniższym wykresie rozkładu widać, że cztery spośród bitstring są wyraźnie bardziej prawdopodobne niż pozostałe.

In [18]:
max_cut_paulis_100 = build_max_cut_paulis(graph_100)

cost_hamiltonian_100 = SparsePauliOp.from_sparse_list(max_cut_paulis_100, 100)
print("Cost Function Hamiltonian:", cost_hamiltonian_100)

Cost Function Hamiltonian: SparsePauliOp(['...'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0

#### Hamiltonian &rarr; quantum circuit

In [19]:
circuit_100 = QAOAAnsatz(cost_operator=cost_hamiltonian_100, reps=1)
circuit_100.measure_all()

circuit_100.draw("mpl", fold=False, scale=0.2, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/9693adfc-0.avif" alt="Output of the previous code cell" />

### Step 2: Optimize problem for quantum execution
To scale the circuit optimization step to utility-scale problems, you can take advantage of the high performance transpilation strategies introduced in Qiskit SDK v1.0. Other tools include the new transpiler service with [AI enhanced transpiler passes](/docs/guides/ai-transpiler-passes).

In [20]:
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

candidate_circuit_100 = pm.run(circuit_100)
candidate_circuit_100.draw("mpl", fold=False, scale=0.1, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3a14e7ad-0.avif" alt="Output of the previous code cell" />

![Wynik poprzedniej komórki kodu](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/650875e9-adbc-43bd-9505-556be2566278-0.avif)

#### Wizualizacja najlepszego cięcia
Na podstawie optymalnego bitstring możesz zwizualizować to cięcie na oryginalnym grafie.

In [21]:
initial_gamma = np.pi
initial_beta = np.pi / 2
init_params = [initial_beta, initial_gamma]

objective_func_vals = []  # Global variable
with Session(backend=backend) as session:
    # If using qiskit-ibm-runtime<0.24.0, change `mode=` to `session=`
    estimator = Estimator(mode=session)

    estimator.options.default_shots = 1000

    # Set simple error suppression/mitigation options
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XY4"
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit_100, cost_hamiltonian_100, estimator),
        method="COBYLA",
    )
    print(result)

 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -3.9939191365979383
       x: [ 1.571e+00  3.142e+00]
    nfev: 29
   maxcv: 0.0


![Wynik poprzedniej komórki kodu](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/33135970-8bc4-4fb2-ab87-08726a432ce4-0.avif)

I oblicz wartość cięcia:

In [22]:
optimized_circuit_100 = candidate_circuit_100.assign_parameters(result.x)
optimized_circuit_100.draw("mpl", fold=False, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/1c432c2e-0.avif" alt="Output of the previous code cell" />

Finally, execute the circuit with the optimal parameters to sample from the corresponding distribution.

In [23]:
# If using qiskit-ibm-runtime<0.24.0, change `mode=` to `backend=`
sampler = Sampler(mode=backend)
sampler.options.default_shots = 10000

# Set simple error suppression/mitigation options
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XY4"
sampler.options.twirling.enable_gates = True
sampler.options.twirling.num_randomizations = "auto"


pub = (optimized_circuit_100,)
job = sampler.run([pub], shots=int(1e4))

counts_int = job.result()[0].data.meas.get_int_counts()
counts_bin = job.result()[0].data.meas.get_counts()
shots = sum(counts_int.values())
final_distribution_100_int = {
    key: val / shots for key, val in counts_int.items()
}

## Część II. Skalowanie w górę!
Masz dostęp do wielu urządzeń z ponad 100 qubitami na platformie IBM Quantum&reg;. Wybierz jedno z nich, aby rozwiązać problem Max-Cut na ważonym grafie z 100 węzłami. To jest problem „skali użytkowej". Kroki budowy przepływu pracy są takie same jak powyżej, ale z dużo większym grafem.

In [24]:
plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/0fda3611-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/590fe2ce-0.avif)

### Krok 1: Odwzorowanie klasycznych danych wejściowych na problem kwantowy
#### Graf &rarr; Hamiltonian
Najpierw przekształć graf, który chcesz rozwiązać, bezpośrednio w Hamiltonian odpowiedni dla QAOA.

In [25]:
_PARITY = np.array(
    [-1 if bin(i).count("1") % 2 else 1 for i in range(256)],
    dtype=np.complex128,
)


def evaluate_sparse_pauli(state: int, observable: SparsePauliOp) -> complex:
    """Utility for the evaluation of the expectation value of a measured state."""
    packed_uint8 = np.packbits(observable.paulis.z, axis=1, bitorder="little")
    state_bytes = np.frombuffer(
        state.to_bytes(packed_uint8.shape[1], "little"), dtype=np.uint8
    )
    reduced = np.bitwise_xor.reduce(packed_uint8 & state_bytes, axis=1)
    return np.sum(observable.coeffs * _PARITY[reduced])


def best_solution(samples, hamiltonian):
    """Find solution with lowest cost"""
    min_cost = 1000
    min_sol = None
    for bit_str in samples.keys():
        # Qiskit use little endian hence the [::-1]
        candidate_sol = int(bit_str)
        # fval = qp.objective.evaluate(candidate_sol)
        fval = evaluate_sparse_pauli(candidate_sol, hamiltonian).real
        if fval <= min_cost:
            min_sol = candidate_sol

    return min_sol


best_sol_100 = best_solution(final_distribution_100_int, cost_hamiltonian_100)
best_sol_bitstring_100 = to_bitstring(int(best_sol_100), len(graph_100))
best_sol_bitstring_100.reverse()

print("Result bitstring:", best_sol_bitstring_100)

Result bitstring: [1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1]


Next, visualize the cut. Nodes of the same color belong to the same group.

In [26]:
plot_result(graph_100, best_sol_bitstring_100)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/b4a25e28-0.avif" alt="Output of the previous code cell" />

#### Hamiltonian &rarr; Circuit kwantowy

In [27]:
cut_value_100 = evaluate_sample(best_sol_bitstring_100, graph_100)
print("The value of the cut is:", cut_value_100)

The value of the cut is: 124


![Output of the previous code cell](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/9693adfc-0.avif)

### Krok 2: Optymalizacja problemu pod kątem wykonania kwantowego
Aby skalować etap optymalizacji układu do problemów skali użytkowej, możesz skorzystać z wysokowydajnych strategii transpilacji wprowadzonych w Qiskit SDK v1.0. Inne narzędzia obejmują nowy serwis Transpilera z [ulepszonymi przejściami transpilacji opartymi na AI](/guides/ai-transpiler-passes).

In [28]:
# auxiliary function to help plot cumulative distribution functions
def _plot_cdf(objective_values: dict, ax, color):
    x_vals = sorted(objective_values.keys(), reverse=True)
    y_vals = np.cumsum([objective_values[x] for x in x_vals])
    ax.plot(x_vals, y_vals, color=color)


def plot_cdf(dist, ax, title):
    _plot_cdf(
        dist,
        ax,
        "C1",
    )
    ax.vlines(min(list(dist.keys())), 0, 1, "C1", linestyle="--")

    ax.set_title(title)
    ax.set_xlabel("Objective function value")
    ax.set_ylabel("Cumulative distribution function")
    ax.grid(alpha=0.3)


# auxiliary function to convert bit-strings to objective values
def samples_to_objective_values(samples, hamiltonian):
    """Convert the samples to values of the objective function."""

    objective_values = defaultdict(float)
    for bit_str, prob in samples.items():
        candidate_sol = int(bit_str)
        fval = evaluate_sparse_pauli(candidate_sol, hamiltonian).real
        objective_values[fval] += prob

    return objective_values

In [29]:
result_dist = samples_to_objective_values(
    final_distribution_100_int, cost_hamiltonian_100
)

Finally, you can plot the cumulative distribution function to visualize how each sample contributes to the total probability distribution and the corresponding objective value. The horizontal spread shows the range of objective values of the samples in the final distribution. Ideally, you would see that the cumulative distribution function has "jumps" at the lower end of the objective function value axis. This would mean that few solutions with low cost have high probability of being sampled. A smooth, wide curve indicates that each sample is similarly likely, and they can have very different objective values, low or high.

In [30]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
plot_cdf(result_dist, ax, "Eagle device")

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/4381a2b3-0.avif" alt="Output of the previous code cell" />

Po znalezieniu optymalnych parametrów z uruchomienia QAOA na urządzeniu przypisz parametry do układu.